# 01 - ETL: Acesso à Energia Elétrica no Amazonas
**Fonte:** Censo IBGE 2022 + dados ANEEL/IEMA por município  
**Projeção:** EPSG:31980 (SIRGAS 2000 / UTM zona 20S)  
**Saída:** `energia-am.geojson` · `municipios-am.geojson`


## 1. Importações

In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import warnings
warnings.filterwarnings('ignore')

print("Bibliotecas carregadas!")


Bibliotecas carregadas!


## 2. Carregando os dados de domicílios por município

O arquivo `energia-am-municipios.csv` contém o total de domicílios por município
e os dados de acesso à energia elétrica baseados no Censo IBGE 2022,
considerando a cobertura do Sistema Interligado Nacional (SIN) e dos
Sistemas Isolados (SISOL) no Amazonas.


In [2]:
energia_mun = pd.read_csv('energia-am-municipios.csv', dtype={'CD_MUN': str})
energia_sit = pd.read_csv('energia-am-rural-urbano.csv', dtype={'CD_MUN': str})

print(f"Municípios carregados: {len(energia_mun)}")
print(f"Combinações município × situação: {len(energia_sit)}")

Municípios carregados: 62
Combinações município × situação: 124


## 3. Resumo geral do Amazonas

In [3]:
total_dom = energia_mun['total_dom'].sum()
total_sem = energia_mun['sem_energia'].sum()
pct_geral = total_sem / total_dom * 100

print("=== Resumo do Amazonas ===")
print(f"Total de domicílios:        {total_dom:>12,.0f}")
print(f"Sem acesso adequado à energia:       {total_sem:>12,.0f}  ({pct_geral:.1f}%)")
print(f"Com acesso adequado à energia:       {total_dom - total_sem:>12,.0f}  ({100 - pct_geral:.1f}%)")

=== Resumo do Amazonas ===
Total de domicílios:           1,077,435
Sem acesso adequado à energia:            114,452  (10.6%)
Com acesso adequado à energia:            962,983  (89.4%)


## 4. Carregando o mapa e reprojetando para EPSG:31980

O GeoJSON do IBGE vem em coordenadas geográficas (EPSG:4326 — latitude/longitude).
Reprojetamos para **EPSG:31980** (UTM zona 20S / SIRGAS 2000) para calcular
distâncias em metros com precisão para a região Norte do Brasil.


In [4]:
mapa_am = gpd.read_file('geojs-13-mun.json')
print(f"Projeção original:   {mapa_am.crs}")

mapa_am = mapa_am.to_crs(epsg=31980)
print(f"Após reprojeção:     {mapa_am.crs}")
print(f"Municípios no mapa:  {len(mapa_am)}")


Projeção original:   EPSG:4326
Após reprojeção:     EPSG:31980
Municípios no mapa:  62


## 5. Join espacial: dados de energia + geometria dos municípios

In [5]:
mapa_final = mapa_am.merge(
    energia_mun, left_on='id', right_on='CD_MUN', how='left'
)
print(f"Municípios com dados: {mapa_final['pct_sem_energia'].notna().sum()} de {len(mapa_final)}")


Municípios com dados: 62 de 62


## 6. Calculando distância até Manaus (km)

Calculamos a distância em quilômetros do centro de cada município até Manaus.
Isso será usado na análise de correlação com o nível de exclusão energética.

In [6]:
manaus_geo = gpd.GeoDataFrame(
    geometry=[Point(-60.0251, -3.1019)], crs='EPSG:4326'
).to_crs(epsg=31980)
manaus_ponto = manaus_geo.geometry.iloc[0]

mapa_final['centroide']      = mapa_final.geometry.centroid
mapa_final['dist_manaus_km'] = mapa_final['centroide'].apply(
    lambda p: round(p.distance(manaus_ponto) / 1000, 1)
)

mapa_final.loc[mapa_final['NM_MUN'] == 'Manaus', 'dist_manaus_km'] = 0.0

print("Mais próximos de Manaus:")
print(mapa_final[['NM_MUN','dist_manaus_km']].sort_values('dist_manaus_km').head(5).to_string(index=False))
print()
print("Mais distantes de Manaus:")
print(mapa_final[['NM_MUN','dist_manaus_km']].sort_values('dist_manaus_km', ascending=False).head(5).to_string(index=False))


Mais próximos de Manaus:
           NM_MUN  dist_manaus_km
           Manaus             0.0
         Iranduba            50.6
Careiro da Várzea            51.6
          Careiro            76.0
 Rio Preto da Eva            76.6

Mais distantes de Manaus:
          NM_MUN  dist_manaus_km
         Guajará          1497.7
         Ipixuna          1344.0
Atalaia do Norte          1335.8
          Envira          1218.3
        Eirunepé          1212.8


## 7. Exportando os arquivos finais

In [7]:
mapa_export = mapa_final.drop(columns=['centroide'])

mapa_export.to_file('municipios-am.geojson', driver='GeoJSON')
mapa_export.to_file('energia-am.geojson',    driver='GeoJSON')

print("Arquivos gerados:")
print("  ✓  municipios-am.geojson")
print("  ✓  energia-am.geojson")
print()
print("=== ETL concluído! ===")


Arquivos gerados:
  ✓  municipios-am.geojson
  ✓  energia-am.geojson

=== ETL concluído! ===
